# Online Retail II — Introduction and Problem Framing

> **Thesis.** This dataset is a transactional ledger, not a modelling table. Unsupervised learning becomes useful only after we define the unit of analysis and engineer features that map commerce into a geometry where similarity has business meaning.

### 1. What this dataset is

A multi-year export from a UK online retailer. Each row is a **line item** on an invoice. The table captures products bought, quantities, prices, timestamps, and basic customer metadata. There is no single “row per customer” or “row per product” view. We must construct it.

#### 1.1 Canonical columns

| Column        | Type      | Typical role                                   | Notes that matter for modelling                              |
|:--------------|:----------|:-----------------------------------------------|:--------------------------------------------------------------|
| `InvoiceNo`   | string    | Transaction grouping key                        | Cancellation invoices often start with `C`.                   |
| `StockCode`   | string    | Product identifier                              | Product taxonomy is implicit, text `Description` is noisy.    |
| `Description` | string    | Human-readable product text                     | Typos, duplicates, subtle variants. Candidate for vectorising.|
| `Quantity`    | integer   | Items on the line                               | Can be negative on reversals. Zero appears in edge cases.     |
| `InvoiceDate` | datetime  | Event time                                      | Time zones and daylight savings must be handled consistently. |
| `UnitPrice`   | float     | Price per unit                                  | Zero or extreme values can occur. Check currency consistency. |
| `CustomerID`  | integer   | Customer identifier                             | Missing for some rows. Decide policy explicitly.              |
| `Country`     | string    | Customer location                               | High-cardinality if expanded. Watch the effect on distances.  |

### 2. Why it is non-trivial

1. **It is a ledger.** Clustering line items is rarely meaningful. We must decide what an **entity** is before analysis.
2. **Reality is messy.** Cancellations, negative quantities, zero prices, duplicates, missing `CustomerID`, and free-text descriptions introduce ambiguity.
3. **Time changes behaviour.** Seasonality, promotions, and drift make global aggregates misleading if time is ignored.
4. **Mixed data types.** Numeric spend and counts live beside countries and noisy text. Off-the-shelf Euclidean distance is fragile without preparation.

### 3. The first decision: unit of analysis

Different questions imply different entities and different feature engineering.

| Goal we care about                 | Entity to represent | Core evidence to aggregate                                      | Typical downstream method                |
|:-----------------------------------|:--------------------|:-----------------------------------------------------------------|:-----------------------------------------|
| Customer segmentation              | Customer            | RFM, basket size, product breadth, cancellation behaviour        | K-means or GMM after scaling or PCA       |
| Product families and substitutions | Product             | Co-purchase networks, seasonality profiles, price dynamics       | Community detection, hierarchical, HDBSCAN|
| Operational rhythms                | Invoice or time bin | Basket composition, value per invoice, intra-day or weekly shape | Time-series clustering, DTW-based methods |
| Anomaly and fraud signals          | Customer or invoice | Rare patterns, abnormal returns, atypical value or timing        | IsolationForest, LOF, One-Class SVM       |


### 4. Feature families that actually transfer to practice

We do not feed raw ledgers to algorithms. We **construct** a feature matrix where distances mean something.

#### 4.1 Behavioural features for customers

| Theme             | Examples that work in practice                                           | Caveats to document                                           |
|:------------------|:-------------------------------------------------------------------------|:--------------------------------------------------------------|
| Intensity         | Total spend, number of invoices, items per basket                        | Heavy tails suggest robust or quantile scaling                |
| Freshness         | Recency in days since last purchase                                      | Anchor to the dataset max date to avoid leakage               |
| Regularity        | Inter-purchase time variability, active months                           | Sensitive to sparse histories                                 |
| Breadth vs focus  | Distinct products count, Herfindahl index over product shares            | Product taxonomy resolution affects interpretation            |
| Returns behaviour | Cancellation rate, value of returns                                      | Separate from core spend to avoid contaminating centroids     |

#### 4.2 Representations for products

- Co-occurrence vectors over baskets, normalised to dampen popularity
- Seasonal signatures over week-of-year or month-of-year
- Price and discount dynamics, volatility measures

#### 4.3 Handling mixed types

- Numeric blocks: scale thoughtfully (Standard, Robust, or Quantile transformers)
- Categorical blocks: one-hot can dominate Euclidean distances if high-dimensional
- Text blocks: TF-IDF or embeddings with cosine similarity, not naive Euclidean

### 5. Pitfalls to avoid

- **Clustering the wrong object.** Line items are not customers. Decide the entity first.
- **Geometry by accident.** Unscaled spend swamps everything. Mixed blocks need tailored treatment.
- **Projection fallacies.** Pretty t-SNE/UMAP plots are not clusters by default. Validate structure independently.
- **Silent coercions.** NaT dates, inferred types, and dropped rows create hidden data loss. Log every decision.
- **Country one-hot overload.** Hundreds of sparse bits distort Euclidean distance unless re-weighted or separated.


### 6. What we are trying to achieve here

1. **Construct auditable feature matrices** from transactional data for a chosen entity.
2. **Define similarity deliberately** so that near points are meaningfully similar in business terms.
3. **Explore latent structure** with clustering and dimensionality reduction without fooling ourselves.
4. **Validate and profile** segments with internal metrics, stability checks, and domain-level summaries.
5. **Translate findings into action**: segment playbooks, product taxonomy hints, anomaly watchlists.


### 7. The working approach we will follow

1. **Load and audit** the raw ledger with explicit typing and date handling.  
2. **Declare the unit of analysis** and the time window for the question at hand.  
3. **Clean with intent**: split cancellation analytics from core spend, remove duplicates, document exclusions.  
4. **Engineer features** that encode behaviour and composition, keeping provenance for every column.  
5. **Shape the geometry**: scale numerics, isolate sparse categoricals, choose metrics that fit representations.  
6. **Only then** apply clustering, embeddings, or anomaly scoring, followed by rigorous validation and profiling.


### 8. Key modelling constraint to remember

> Algorithms consume matrices, not ledgers. The quality of unsupervised results is determined less by the choice of algorithm and more by the **mapping from domain behaviour to feature space**. We will make that mapping explicit at every step.


In [19]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import pandas as pd

In [20]:
url = "https://github.com/jhlopesalves/data-science-notebooks/raw/refs/heads/main/Python/projects/online_retail_ii/data/online_retail_II.xlsx"

In [21]:
dtype_map = {
    "Invoice": "string",
    "StockCode": "string",
    "Description": "string",
    "Quantity": "Int64",
    "InvoiceDate": "string",
    "Price": "float64",
    "Customer ID": "Int64",
    "Country": "string",
}

In [22]:
retail = pd.read_excel(url, dtype=dtype_map)

In [23]:
retail.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom


In [24]:
retail["InvoiceDate"] = pd.to_datetime(retail["InvoiceDate"], errors="coerce", utc=True)
retail["Customer ID"] = retail["Customer ID"].astype("category")

In [25]:
retail.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 525461 entries, 0 to 525460
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype              
---  ------       --------------   -----              
 0   Invoice      525461 non-null  string             
 1   StockCode    525461 non-null  string             
 2   Description  522533 non-null  string             
 3   Quantity     525461 non-null  Int64              
 4   InvoiceDate  525461 non-null  datetime64[ns, UTC]
 5   Price        525461 non-null  float64            
 6   Customer ID  417534 non-null  category           
 7   Country      525461 non-null  string             
dtypes: Int64(1), category(1), datetime64[ns, UTC](1), float64(1), string(4)
memory usage: 29.7 MB


In [26]:
retail.describe()

,Quantity,Price
count,525461.0,525461.000000
mean,10.337667,4.688834
std,107.42411,146.126914
min,-9600.0,-53594.360000
25%,1.0,1.250000
50%,3.0,2.100000
75%,10.0,4.210000
max,19152.0,25111.090000


In [27]:
n_rows, n_cols = retail.shape
missing_summary = retail.isna().mean().sort_values(ascending=False)

n_rows, n_cols, missing_summary.head(10)

(525461,
 8,
 Customer ID    0.205395
 Description    0.005572
 StockCode      0.000000
 Invoice        0.000000
 Quantity       0.000000
 InvoiceDate    0.000000
 Price          0.000000
 Country        0.000000
 dtype: float64)

In [28]:
def min_max(column: pd.DataFrame):
    column_min = column.min()
    column_max = column.max()
    return f"Min: {column_min}, Max: {column_max}"

In [30]:
retail["Price"] = np.abs(retail["Price"])
retail["Quantity"] = np.abs(retail["Quantity"])

In [ ]:
quantity_range = min_max(retail["Quantity"])
print(quantity_range)

Min: 1, Max: 19152


In [32]:
price_range = min_max(retail["Price"])

print(price_range)

Min: 0.0, Max: 53594.36


In [33]:
essential = [
    "Invoice",
    "StockCode",
    "InvoiceDate",
    "Quantity",
    "Price",
    "Customer ID",
]

retail = retail.dropna(subset=essential)

In [34]:
retail["is_cancellation"] = retail["Invoice"].str.upper().str.startswith("C")
retail["line_total"] = retail["Quantity"].astype("int64") * retail["Price"]

In [35]:
mask_core = (retail["Quantity"] > 0) & (retail["Price"] > 0)
retail = retail.loc[mask_core]

In [36]:
retail

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,is_cancellation,line_total
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00+00:00,6.95,13085,United Kingdom,False,83.40
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00+00:00,6.75,13085,United Kingdom,False,81.00
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00+00:00,6.75,13085,United Kingdom,False,81.00
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00+00:00,2.10,13085,United Kingdom,False,100.80
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00+00:00,1.25,13085,United Kingdom,False,30.00
...,...,...,...,...,...,...,...,...,...,...
525456,538171,22271,FELTCRAFT DOLL ROSIE,2,2010-12-09 20:01:00+00:00,2.95,17530,United Kingdom,False,5.90
525457,538171,22750,FELTCRAFT PRINCESS LOLA DOLL,1,2010-12-09 20:01:00+00:00,3.75,17530,United Kingdom,False,3.75
525458,538171,22751,FELTCRAFT PRINCESS OLIVIA DOLL,1,2010-12-09 20:01:00+00:00,3.75,17530,United Kingdom,False,3.75
525459,538171,20970,PINK FLORAL FELTCRAFT SHOULDER BAG,2,2010-12-09 20:01:00+00:00,3.75,17530,United Kingdom,False,7.50


In [40]:
len(retail["Description"].value_counts().unique())

527

In [ ]:
len(retail["Customer ID"].value_counts().unique())